In [ ]:
try:
    from hillclimbers import climb_hill, partial
except:
    !pip install -q hillclimbers
    from hillclimbers import climb_hill, partial

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'

In [ ]:
import os, numpy as np, pandas as pd
from scipy.stats import rankdata

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from time import time, sleep

from sklearn.svm import SVC, LinearSVC
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.linear_model import (
        LogisticRegressionCV, ElasticNetCV, RidgeCV, LassoCV,
        LogisticRegression, LinearRegression,
        Ridge, BayesianRidge, ElasticNet, Lasso,
)

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, r2_score, PredictionErrorDisplay
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler, MinMaxScaler, QuantileTransformer, FunctionTransformer

import warnings
warnings.simplefilter('ignore')
warnings.filterwarnings('ignore')

## -- Global Settings --
# sklearn.set_config(transform_output="pandas")

# pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', 100)
sns.set_style("whitegrid")
# plt.style.use("ggplot")

warnings

In [ ]:
TEAL, GOLD, CORAL, BLUE, DARK, CRIM, MUTED, INK, PAPER = '#168c91','#e8b44f','#dc705a',\
                                                         'dodgerblue','#0B4F7A','crimson',\
                                                         '#627D98','#102A43','#faf9f5'
plt.rcParams.update({
    'figure.facecolor': PAPER,
    'axes.facecolor': PAPER,
    'savefig.facecolor': PAPER,
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.titleweight': 'semibold',
    'axes.labelcolor': INK,
    'text.color': INK,
    'xtick.color': DARK,
    'ytick.color': DARK,
    'axes.edgecolor': MUTED,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.color': MUTED,
    'patch.edgecolor': DARK,
    # 'patch.force_edgecolor': True,
    # 'patch.linewidth': 1.5,
    # 'figure.dpi':110,
    # 'savefig.dpi':160,
})
# CMAP = LinearSegmentedColormap.from_list('atlas', ['#f0f5ef','#a9d6ca',TEAL,INK])
sns.set_palette([TEAL, GOLD, CORAL, BLUE, MUTED, CRIM, DARK, INK, PAPER])

## -- Set Global Seed --
CFG = {
    'FOLDS':  5,
    'SEED':   42,
    'GREEN':  '\033[32m',
    'YELLOW': '\033[33m',
    'RED':    '\033[31m',
    'RESET':  '\033[0m',
}

print(f"CLASSIC {CFG['GREEN']} GREEN {CFG['RESET']}", end="")
print(f"{CFG['YELLOW']}YELLOW {CFG['RESET']} {CFG['RED']} RED {CFG['RESET']}")

In [ ]:
def prob_to_logit(p, eps=1e-15, clip=30.0):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, eps, 1.0 - eps)
    z = np.log(p / (1.0 - p))
    return np.clip(z, -clip, clip)

def ordinal_common_fold_ranks(values, common_size):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    n = values.size
    if n == 0:
        return np.empty(0, dtype=np.int32)
    if common_size <= 1:
        return np.zeros(n, dtype=np.int32)
    order = np.argsort(values, kind="mergesort")
    pos = np.arange(n, dtype=np.int64)
    ranks_sorted = np.clip((pos * common_size) // n, 0, common_size - 1).astype(np.int32)
    ranks = np.empty(n, dtype=np.int32)
    ranks[order] = ranks_sorted
    return ranks

def fold_calibrate_oof_probs(oof, folds, eps=1e-15):
    oof = np.asarray(oof, dtype=np.float64).reshape(-1)
    min_fold_size = int(min(len(va_idx) for _, va_idx in folds))
    ranks = np.empty(oof.shape[0], dtype=np.int32)
    for _, va_idx in folds:
        ranks[va_idx] = ordinal_common_fold_ranks(oof[va_idx], common_size=min_fold_size)
    rank_means = pd.DataFrame({"rank": ranks, "prob": oof}).groupby("rank", sort=True)["prob"].mean()
    calibrated = rank_means.iloc[ranks].to_numpy(dtype=np.float64)
    return np.clip(calibrated, eps, 1.0 - eps)

# - For Pipeline, use with FunctionTransformer(func=rank_transform)
def rank_transform(X):
    return rankdata(np.asarray(X), axis=0) / len(X)
    # return X.rank(method='average', pct=True)

print("⚙️ Helper functions ready! ⚙️")

# 1.__LOAD DATA

In [ ]:
## -- Load Data --
PATH = "/kaggle/input/competitions/playground-series-s6e9/"
submit = pd.read_csv(PATH+"sample_submission.csv")
train = pd.read_csv(PATH+"train.csv")

if "id" in train.columns:
    train = train.drop(columns=["id"])

TARGET = train.columns[-1]
y_true = train[TARGET].map({'Yes': 1, 'No': 0}) #.to_numpy()

print(train.shape, type(train))
train.head()

In [ ]:
# ## -- Append predicted columns as extra features --
# ## -- os.walk: Deep search with subfolders --
# def dataloader(filepath, train_prefix, test_prefix):    
#     train_df = pd.DataFrame()
#     test_df = pd.DataFrame()

#     print('Loading .npy files: ', end='')
#     for (root, dirs, files) in os.walk(filepath):
#         for i, file in tqdm(enumerate(sorted(files), 1), total=len(files)):
#             # if i%10 == 0: print(f'{i}%.. ', end='')
#             if file.endswith('.npy') and train_prefix in file:
#                 train_path = np.load(os.path.join(root, file))
#                 train_df = pd.concat([train_df, pd.Series(train_path, name=file[4:-4])], axis=1)
#             elif file.endswith('.npy') and test_prefix in file:
#                 test_path = np.load(os.path.join(root, file))
#                 test_df = pd.concat([test_df, pd.Series(test_path, name=file[5:-4])], axis=1)
#     print()

#     print('Loading .parquet files: ', end='')
#     for (root, dirs, files) in os.walk(filepath):
#         for i, file in tqdm(enumerate(sorted(files), 1), total=len(files)):
#             # if i%10 == 0: print(f"{i}%.. ", end="")
#             if file.endswith('.parquet') and train_prefix in file:
#                 train_path = os.path.join(root, file)
#                 train_df = pd.concat([train_df, pd.read_parquet(train_path)], axis=1)
#             elif file.endswith('.parquet') and test_prefix in file:
#                 test_path = os.path.join(root, file)
#                 test_df = pd.concat([test_df, pd.read_parquet(test_path)], axis=1)
#     print()
#     print(f"OOF  Shape: {train_df.shape}")
#     print(f"TEST Shape: {test_df.shape}")
#     print()

#     return train_df, test_df

# def rename_duplicate_cols(df: pd.DataFrame) -> pd.DataFrame:
#     original_cols = list(df.columns)
#     new_cols = []
#     # Track how many times we've seen each original name
#     seen = {}

#     for col in tqdm(original_cols, desc="Fixing columns"):
#         if col not in seen:
#             # First occurrence: keep name unchanged
#             seen[col] = 1
#             new_cols.append(col)
#         else:
#             # Duplicate: append suffix _k where k starts at 0
#             suffix = seen[col] - 1 # 0 for first duplicate, 1 for second,
#             new_name = f"{col}_{suffix}"
#             new_cols.append(new_name)
#             seen[col] += 1

#     # Apply new column names
#     df_renamed = df.copy()
#     df_renamed.columns = new_cols

#     return df_renamed

# print("Load functions ready!")

In [ ]:
# PATHS = [
#     "/kaggle/input/datasets/",
# ]

# ## -- Load dataframes --
# oofs = []
# preds  = []

# ## -- Load train/test datasets --
# for p in PATHS:
#     tr, ts = dataloader(p, 'oof', 'test')
#     oofs.append(tr)
#     preds.append(ts)

# ## -- Concat datasets --
# oof_df  = pd.concat(oofs, axis=1)
# pred_df = pd.concat(preds, axis=1)

# # ## -- Drop models if necessary --
# # for c in oof_df.columns:
# #     if 'cvStack' in c or 'L3' in c: #
# #         oof_df.drop([c], axis=1, inplace=True)
# #         pred_df.drop([c], axis=1, inplace=True)
# #     elif c == TARGET:
# #         oof_df.drop([c], axis=1, inplace=True)

# print('-'*50)
# ## -- Check for missingness --
# print(f"OOF Null: {oof_df.isna().sum().sum()}", end=" | ")
# print(f"TEST Null: {pred_df.isna().sum().sum()}")
# print('-'*50)

# ## -- Apply column rename function --
# oof_df = rename_duplicate_cols(oof_df)
# pred_df = rename_duplicate_cols(pred_df)

# print()
# print(f"{CFG['YELLOW']}••••• Updated columns: {oof_df.shape[1]}")

# display(oof_df.head(3))
# print()
# display(pred_df.head(3))

In [ ]:
# import os
# import numpy as np
# import pandas as pd
## --Config settings --

FOLDS = 5
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)
sk_FOLDS = list(skf.split(np.zeros(len(y_true), dtype=np.int8), y_true))

data_path = "/kaggle/input/datasets/sylvesterdalmeida/ps6e9-oof-outputs"

train_dict = {}
test_dict = {}

# Load Train Files
print("🟦 Loading train files: ", end="")
for dirname, _, filenames in os.walk(data_path):
    for i, filename in enumerate(sorted(filenames), 1):
        if filename.startswith("oof"):
            file = os.path.join(dirname, filename)
            col_name = filename.split("oof_")[1].replace(".npy", "")
            train_dict[col_name] = np.load(file).ravel()
            print('•', end="")

# Load Test Files
print("\n\n🟩 Loading test  files: ", end="")
for dirname, _, filenames in os.walk(data_path):
    for i, filename in enumerate(sorted(filenames), 1):
        if filename.startswith("test"):
            file = os.path.join(dirname, filename)
            col_name = filename.split("test_")[1].replace(".npy", "")
            test_dict[col_name] = np.load(file).ravel()
            print('•', end="")

print('\n'+'-'*50)

oof_mismatch = []
for k, v in train_dict.items():
    # print(f"{k:>25}", v.shape)
    if len(v) != len(train):
        oof_mismatch.append(k)

test_mismatch = []
for k, v in test_dict.items():
    # print(f"{k:>25}", v.shape)
    if len(v) != len(submit):
        test_mismatch.append(k)

print('\n'+'-'*50)
print(f"oof  mimatch: {oof_mismatch}")
print(f"test mimatch: {test_mismatch}")
print(f"Dropping all columns with mismatches: train={len(oof_mismatch)}, test={len(test_mismatch)}")
new_train_dict = {k: v for k, v in train_dict.items() if k not in oof_mismatch+test_mismatch}
new_test_dict  = {k: v for k, v in test_dict.items() if k not in oof_mismatch+test_mismatch}

# Create DataFrames separately
oof_df  = pd.DataFrame(new_train_dict)
pred_df = pd.DataFrame(new_test_dict)

MODEL_COLS = [*oof_df.columns]

# # -- Calibrate if needed --
# print(f"\n\n{CFG['YELLOW']}Calibrating models", end="... ")
# for col in MODEL_COLS:
#     print(f"• ", end="")
#     oof_df[col] = fold_calibrate_oof_probs(oof_df[col].to_numpy(), sk_FOLDS)
# print(f"Finished calibration! {CFG['RESET']}")

print('-'*40)
print(f"Train shape: {oof_df.shape} | missing: {oof_df.isna().sum().sum()}")
print(f"Test  shape: {pred_df.shape} | missing: {pred_df.isna().sum().sum()}")
print('-'*40)

display(oof_df.head())
display(pred_df.head())

In [ ]:
sub_ = oof_df.iloc[:5, :5]

_rank = rankdata(np.asarray(sub_), axis=0)
scaled_rank = rankdata(np.asarray(sub_), axis=0) / len(sub_)

print(_rank)
print()
print(scaled_rank)
print()
print(sub_.rank(method='average', pct=True).to_numpy())

In [ ]:
%%time

sub_ = oof_df.sample(n=20, axis="columns") # select by columns

# if len(sub_.columns) > 1:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 20))
sns.heatmap(sub_.corr("spearman"), annot=True, fmt=".4g", cbar=False, ax=ax1)
ax1.set_title("Heatmap Correlation (Spearman)", fontsize=14, fontweight='bold')

sns.ecdfplot(data=sub_, linewidth=1, ax=ax2) # legend=False, 
ax2.set_title("CDF Comparison of Probability Distributions", fontsize=14, fontweight='bold')
ax2.set_xlabel("Probability Value")
ax2.set_ylabel("Cumulative Proportion (CDF)") 
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig("oof_heatmap_cdf_plot")
plt.show()

In [ ]:
# ## -- Append predicted columns as extra features --
# ## -- os.walk: Deep search with subfolders --
# def dataloader(filepath, train_prefix, test_prefix):    
#     train_df = pd.DataFrame()
#     test_df = pd.DataFrame()
    
#     print('Loading .npy files: ', end='')
#     for (root, dirs, files) in os.walk(filepath):
#         for i, file in tqdm(enumerate(sorted(files), 1), total=len(files)):
#             # if i%10 == 0: print(f'{i}%.. ', end='')
#             if file.endswith('.npy') and train_prefix in file:
#                 train_path = np.load(os.path.join(root, file))
#                 train_df = pd.concat([train_df, pd.Series(train_path, name=file[4:-4])], axis=1)
#             elif file.endswith('.npy') and test_prefix in file:
#                 test_path = np.load(os.path.join(root, file))
#                 test_df = pd.concat([test_df, pd.Series(test_path, name=file[5:-4])], axis=1)
#     print()

#     print('Loading .parquet files: ', end='')
#     for (root, dirs, files) in os.walk(filepath):
#         for i, file in tqdm(enumerate(sorted(files), 1), total=len(files)):
#             # if i%10 == 0: print(f"{i}%.. ", end="")
#             if file.endswith('.parquet') and train_prefix in file:
#                 train_path = os.path.join(root, file)
#                 train_df = pd.concat([train_df, pd.read_parquet(train_path)], axis=1)
#             elif file.endswith('.parquet') and test_prefix in file:
#                 test_path = os.path.join(root, file)
#                 test_df = pd.concat([test_df, pd.read_parquet(test_path)], axis=1)
#     print()
#     print(f"OOF  Shape: {train_df.shape}")
#     print(f"TEST Shape: {test_df.shape}")
#     print()

#     return train_df, test_df

# def rename_duplicate_cols(df: pd.DataFrame) -> pd.DataFrame:
#     original_cols = list(df.columns)
#     new_cols = []
#     # Track how many times we've seen each original name
#     seen = {}

#     for col in tqdm(original_cols, desc="Fixing columns"):
#         if col not in seen:
#             # First occurrence: keep name unchanged
#             seen[col] = 1
#             new_cols.append(col)
#         else:
#             # Duplicate: append suffix _k where k starts at 0
#             suffix = seen[col] - 1 # 0 for first duplicate, 1 for second,
#             new_name = f"{col}_{suffix}"
#             new_cols.append(new_name)
#             seen[col] += 1

#     # Apply new column names
#     df_renamed = df.copy()
#     df_renamed.columns = new_cols

#     return df_renamed


In [ ]:
# PATHS = [
#     '/kaggle/input/datasets/',
# ]

# ## -- Load dataframes --
# oofs = []
# preds  = []

# ## -- Load train/test datasets --
# for p in PATHS:
#     tr, ts = dataloader(p, 'oof', 'test')
#     oofs.append(tr)
#     preds.append(ts)

# ## -- Concat datasets --
# oof_df  = pd.concat(oofs, axis=1)
# pred_df = pd.concat(preds, axis=1)

# # ## -- Drop models if necessary --
# # for c in oof_df.columns:
# #     if 'cvStack' in c or 'L3' in c: #
# #         oof_df.drop([c], axis=1, inplace=True)
# #         pred_df.drop([c], axis=1, inplace=True)
# #     elif c == TARGET:
# #         oof_df.drop([c], axis=1, inplace=True)

# print('-'*50)
# ## -- Check for missingness --
# print(f'OOF  Null: {oof_df.isna().sum().sum()} | ',end='')
# print(f'TEST Null: {pred_df.isna().sum().sum()}')
# print('-'*50)

# ## -- Apply column rename function --
# oof_df = rename_duplicate_cols(oof_df)
# pred_df = rename_duplicate_cols(pred_df)

# print()
# print(f'{CFG['YELLOW']}••••• Updated columns: {oof_df.shape[1]}')

# display(oof_df.head(3))
# print()
# display(pred_df.head(3))

In [ ]:
[oof_df.columns.tolist().index(c) for c in oof_df.columns if c not in pred_df.columns]

In [ ]:
## -- Confirm feature name consistency --
for (a, b) in zip(oof_df.columns, pred_df.columns):
    if a == b:
        print(f"{CFG['GREEN']}✓ ", end="")
    else:
        print(f"{CFG['RED']}✘ ", end="")

In [ ]:
def train_best_model(X, X_test, y, clf="logit", log_metric="neg_log_loss",
                     convert_logits=False, use_scaler=False):
    all_models = X.columns
    
    if convert_logits:
        print("+++ Transform probs to logits...")
        X = pd.DataFrame(prob_to_logit(X.to_numpy().astype(np.float32)), columns=all_models)
        X_test = pd.DataFrame(prob_to_logit(X_test.to_numpy().astype(np.float32)), columns=all_models)

    if clf == "logit":
        print("\n+++ Training Logistic Regression +++")
        if use_scaler:
            model = make_pipeline(
                StandardScaler(),
                LogisticRegression(max_iter=10_000, n_jobs=-1, random_state=CFG['SEED'])
            )
            param_grid = {'logisticregression__C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
        else:
            model = LogisticRegression(max_iter=10_000, n_jobs=-1, random_state=CFG['SEED'])
            param_grid = {'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}

        pipe_model = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            cv=sk_FOLDS,
            scoring=log_metric,
            n_jobs=-1,
            # verbose=2,
        ).fit(X.to_numpy(), y.values)
    elif clf == "ridge":
        print("\n+++ Training Ridge +++")
        if use_scaler:
            model = make_pipeline(
                StandardScaler(),
                Ridge(random_state=CFG['SEED'])
            )
            param_grid = {"ridge__alpha": [0.1, 1.0, 10.0, 100.0, 1000.0]}
        else:
            model = Ridge(random_state=CFG['SEED'])
            param_grid = {"alpha": [0.1, 1.0, 10.0, 100.0, 1000.0]}

        pipe_model = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            cv=sk_FOLDS,
            scoring="neg_mean_squared_error",
            n_jobs=-1,
            # verbose=2,
        ).fit(X.to_numpy(), y.values)
    
    cv_scores  = pd.DataFrame(pipe_model.cv_results_).loc[:, "params":]
    best_score = np.round(np.abs(pipe_model.best_score_), 6)
    best_value = np.abs([*pipe_model.best_params_.values()][0])
    display(cv_scores)

    # --- Train full data --
    best_model = pipe_model.best_estimator_

    test_preds_final = []
    full_fit_scores  = []
    full_fit_coefs   = []

    print("="*50)
    print(f"\nTRAINING FULL DATA")
    print("="*50)
    for k in range(10):
        full_data = pd.concat([X, y], axis=1).sample(frac=1.0, random_state=42+k)
        X_2 = full_data.iloc[:, :-1]
        y_2 = full_data.iloc[:, -1]
        if clf == "logit":
            print(f"{clf} --> Full-data fit {k+1}/10 ...", end=" ")
            best_model.fit(X_2.to_numpy(), y_2.values)
            test_preds_final.append(best_model.predict_proba(X_test)[:, 1])
            auc = roc_auc_score(y_2, best_model.predict_proba(X_2.to_numpy())[:, 1])
            print(f"{auc:.6f}")
            full_fit_scores.append(auc)
            if use_scaler:
                full_fit_coefs.append(best_model[-1].coef_[0])
            else:
                full_fit_coefs.append(best_model.coef_[0])
        elif clf == "ridge":
            print(f"{clf} --> Full-data fit {k+1}/10 ...", end=" ")
            best_model.fit(X_2.to_numpy(), y_2.values)
            test_preds_final.append(best_model.predict(X_test))
            auc = roc_auc_score(y_2, best_model.predict(X_2.to_numpy()))
            print(f"{auc:.6f}")
            full_fit_scores.append(auc)
            if use_scaler:
                full_fit_coefs.append(best_model[-1].coef_)
            else:
                full_fit_coefs.append(best_model.coef_)

    test_preds_final = np.mean(test_preds_final, axis=0)
    full_fit_score   = np.round(np.mean(full_fit_scores, axis=0), 6)
    full_fit_coefs   = np.mean(full_fit_coefs, axis=0)

    print("-"*40)
    print(f"Full_fit_score: {full_fit_score}")
    print(f"Best CV score:  {best_score}")
    print(f"Best Params:    {pipe_model.best_params_}")
    print("-"*40)

    full_fit_coef_df = pd.Series(
        dict(zip(all_models, full_fit_coefs)),
        name="abs_coefs").abs().sort_values(ascending=False)[:50]
    
    plt.figure(figsize=(16, max(6, len(full_fit_coef_df) * 0.3)))
    sns.barplot(full_fit_coef_df, orient="h")
    plt.title(f"{clf}: Full-sanity coefs (absolute) weights", fontdict={'size': 14, 'weight': 'semibold'})
    plt.savefig("ridge_oof_coef")
    plt.tight_layout()
    plt.show()

    return test_preds_final, full_fit_score

print("Stacker trainer ready!")

In [ ]:
# model_type = "ridge"
# log_metric = "neg_log_loss"

# test_preds_final, full_fit_score = train_best_model(
#     oof_df,
#     pred_df,
#     y_true,
#     clf=model_type,
#     log_metric=log_metric,
#     # convert_logits=True,
#     # use_scaler=True,
# )

# n = f"submit_{model_type}_full_{full_fit_score}_.csv"

# submit[TARGET] = test_preds_final
# submit.to_csv(n, index=False)
# print(f"\n+++++ Submit file saved as: {n}")

# submit

In [ ]:
# X_meta_train = oof_df.copy()
# X_meta_test  = pred_df.copy()
# y_meta_train = y_true.values

# print(f"{CFG['YELLOW']}Converting probas to logits...{CFG['RESET']}", end="")
# for col in MODEL_COLS:
#     print(col, end="... ")
#     X_meta_train[col] = prob_to_logit(X_meta_train[col].values.astype(np.float32))
#     X_meta_test[col]  = prob_to_logit(X_meta_test[col].values.astype(np.float32))

# print(f"{CFG['YELLOW']} Finished!")

In [ ]:
# %%time

# base_lr = LogisticRegression(max_iter=10_000, n_jobs=-1, random_state=CFG['SEED'])
# pipe = make_pipeline(StandardScaler(), base_lr)

# param_grid = {'logisticregression__C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}

# pipe_model = GridSearchCV(
#     estimator=pipe,
#     param_grid=param_grid,
#     cv=sk_FOLDS,
#     scoring="roc_auc",
#     n_jobs=-1,
#     verbose=1,
# )

# pipe_model.fit(X_meta_train, y_meta_train)

# cv_scores  = pd.DataFrame(pipe_model.cv_results_).loc[:, "params":]
# best_score = np.round(pipe_model.best_score_, 6)
# best_value = [*pipe_model.best_params_.values()][0]
# display(cv_scores)

# print("-"*40)
# print(f"Best Score:  {best_score}")
# print(f"Best Params: {pipe_model.best_params_}")
# print("-"*40)

In [ ]:
# ----------------------------------------
# Shape of scores array: (5, 5)
# Best C value: [0.01]
# ----------------------------------------
# C: 0.001 ---> Mean CV Score: 0.969451
# C:  0.01 ---> Mean CV Score: 0.969475
# C:   0.1 ---> Mean CV Score: 0.969474
# C:   1.0 ---> Mean CV Score: 0.969474
# C:  10.0 ---> Mean CV Score: 0.969474
# CPU times: user 4.59 s, sys: 2.95 s, total: 7.54 s
# Wall time: 1min 7s

In [ ]:
from sklearn.inspection import permutation_importance

def plot_permutation_importance(
    model,
    X_test,
    y_test,
    features=MODEL_COLS,
    metric="roc_auc",
    n_repeats=10,
    seed=42,
    top_n=20,
):
    """
    Computes permutation importance for each column, aggregates class columns 
    belonging to the same model by stripping the class suffix ('_c0', '_c1', '_c2'), 
    and plots the overall importance per model.
    """
    if features is None:
        features = list(X_test.columns) if hasattr(X_test, 'columns') else \
                        [f"Feature_{i}" for i in range(X_test.shape[1])]

    result = permutation_importance(
        model,
        X_test,
        y_test,
        scoring=metric,
        n_repeats=n_repeats,
        random_state=seed,
        n_jobs=-1,
    )

    print("Done!"); print()

    # Create initial DataFrame with raw feature names
    importance_df = pd.DataFrame({
        'Feature': features,
        'Importance_Mean': result.importances_mean,
        'Importance_Std': result.importances_std
    }).sort_values(by='Importance_Mean', ascending=False)[:top_n]

    # Plot aggregated model importances
    plt.figure(figsize=(16, max(6, len(importance_df) * 0.4)))
    ax = sns.barplot(
        data=importance_df,
        x='Importance_Mean',
        y='Feature',
        palette='plasma'
    )

    # Add error bars for standard deviation
    plt.errorbar(
        x=importance_df['Importance_Mean'],
        y=np.arange(len(importance_df)),
        xerr=importance_df['Importance_Std'],
        fmt='none',
        ecolor='black',
        capsize=3,
        alpha=0.6
    )

    # Add numeric value annotations to the right of each bar
    max_val  = max(importance_df['Importance_Mean'])
    mean_std = zip(importance_df['Importance_Mean'], importance_df['Importance_Std'])
    for i, (mean, std) in enumerate(mean_std):
        plt.text(
            mean + (max_val * 0.01), 
            i, 
            f"{mean:.4f}", 
            va='center', 
            fontsize=9
        )

    plt.title(f"Aggregated Model Permutation Importance ({metric})", fontsize=14, fontweight='bold')
    plt.xlabel(f"Total decrease in {metric}")
    plt.ylabel("Models")
    plt.tight_layout()
    plt.savefig("oof_features_permutation_plot")
    plt.show()

    return importance_df 

print("plot_permutation function ready!")

In [ ]:
%%time

perm_clf = LogisticRegression(
    max_iter=10_000,
    C=0.1,
    n_jobs=-1,
    random_state=101,
)

X_train, X_test, y_train, y_test = train_test_split(
    prob_to_logit(oof_df.to_numpy()),
    y_true,
    stratify=y_true,
    random_state=101,
)

# --------------------------------------------------------
perm_clf = make_pipeline(*[
    # FunctionTransformer(func=rank_transform),
    StandardScaler(),
    perm_clf,
])
# X_train = X_train.rank(method="average", pct=True)
# X_test  = X_test.rank(method="average", pct=True)
# --------------------------------------------------------
print(f" ⧗ Fitting {perm_clf[-1]}... ", end="")
perm_clf.fit(X_train, y_train)

print("Done!")

print(" ⧗ Computing permutation importance... ", end="")
perm_imp = plot_permutation_importance(
    perm_clf,
    X_test,
    y_test,
    features=MODEL_COLS,
    metric="roc_auc",
    n_repeats=10,
    top_n=50,
)

In [ ]:
# X_meta_train

In [ ]:
# !rm -r /kaggle/working

In [ ]:
y_true.value_counts(normalize=True)

## ENSEMBLE 1: STACKING

In [ ]:
def train_meta(
    oof_df,
    pred_df,
    y_true,
    clf='model',
    skfold=sk_FOLDS,
    alpha_C=0.01,
    features=None,
    to_logits=False,
    rank_first=False,
    apply_scaler=False,
    verbose=True,
):

    if features is None:
        features = oof_df.columns

    oofs  = oof_df[features].copy()
    preds = pred_df[features].copy()

    oof_preds  = np.zeros(len(oofs))
    test_preds = np.zeros(len(preds))
    
    n_splits   = len(skfold)

    val_scores = []
    oof_alphas = []

    print(f"++++ {clf.title()} (alpha_C={alpha_C}) {n_splits}-Fold CV | Total Models: {len(features)} ++++")
    t0 = time()

    if to_logits:
        print("  ++ Probs to logits...")
        for col in features:
            oofs[col]  = prob_to_logit(oofs[col].to_numpy()) #.astype(np.float32)
            preds[col] = prob_to_logit(preds[col].to_numpy()) #.astype(np.float32)
    for idx, (trn_idx, val_idx) in enumerate(skfold, 1):
        X_trn, y_trn = oofs.iloc[trn_idx], y_true.iloc[trn_idx]
        X_val, y_val = oofs.iloc[val_idx], y_true.iloc[val_idx]
        X_test = preds.copy()

        # if rank_first:
        #     for col in features:
        #         X_trn[col] = rankdata(X_trn[col].to_numpy())
        #         X_test[col] = rankdata(X_test[col].to_numpy())

        #         scaler = StandardScaler()
        #         X_trn[col] = scaler.fit_transform(X_trn[[col]], y_trn)
        #         X_test[col] = scaler.transform(X_test[[col]])

        # --------------------------------------------------------------------------
        if rank_first:
            if idx == 1:
                print("  ++ Ranking models... ")
            X_trn  = X_trn.rank(method='average', pct=True)
            X_val  = X_val.rank(method='average', pct=True)
            X_test = X_test.rank(method='average', pct=True)

        if apply_scaler:
            if idx == 1:
                print("  ++ Scaling features... ")
            scaler = StandardScaler().set_output(transform="pandas")
            X_trn  = scaler.fit_transform(X_trn, y_true)
            X_val  = scaler.transform(X_val)
            X_test = scaler.transform(X_test)
        # --------------------------------------------------------------------------

        if verbose and idx == 1:
            print(f"train={X_trn.shape}, valid={X_val.shape}, test={X_test.shape}")

        if clf == "mlp":
            model = MLPClassifier(
                alpha=alpha_C, # default=0.0001
                max_iter=50,
                tol=0.0001,
                hidden_layer_sizes=(X_trn.shape[1],), # default=(100,)
                random_state=101,
                # batch_size=128, # default='auto'
                activation='relu', # {'identity', 'logistic', 'tanh', 'relu'}, default='relu'
                solver='adam', # {'lbfgs', 'sgd', 'adam'}, default='adam'
                learning_rate='constant',
                learning_rate_init=0.001, # default=0.001
                power_t=0.5,
                shuffle=True,
                verbose=False,
                warm_start=False,
                momentum=0.9,
                nesterovs_momentum=True,
                early_stopping=True, # default=False
                validation_fraction=0.1,
                beta_1=0.9,
                beta_2=0.999,
                epsilon=1e-08,
                n_iter_no_change=10,
                max_fun=15000,
            )
            model.fit(X_trn, y_trn)
            oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
            test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits
            # fold_score = roc_auc_score(y_val, oof_preds[val_idx])
            # val_scores.append(fold_score)
            # if verbose:
            #     print(f"Fold {idx}/{n_splits}: {fold_score:.6f}", end=" | ")
        elif clf == "bayes":
            model = BayesianRidge(max_iter=1000, tol=0.0001)
            model.fit(X_trn, y_trn)
            oof_preds[val_idx] = model.predict(X_val)
            test_preds += model.predict(X_test) / skf.n_splits
            # fold_score = roc_auc_score(y_val, oof_preds[val_idx])
            # val_scores.append(fold_score)
            # if verbose:
            #     print(f"Fold {idx}/{n_splits}: {fold_score:.6f}", end=" | ")
        elif clf == "linear":
            model = LinearRegression(n_jobs=-1) 
            model.fit(X_trn, y_trn)
            oof_preds[val_idx] = model.predict(X_val)
            test_preds += model.predict(X_test) / skf.n_splits
            # fold_score = roc_auc_score(y_val, oof_preds[val_idx])
            # val_scores.append(fold_score)
            # if verbose:
            #     print(f"Fold {idx}/{n_splits}: {fold_score:.6f}", end=" | ")
        elif clf == "elastic":
            model = ElasticNet(
                alpha=alpha_C,
                max_iter=10_000,
                random_state=101,
            )
            model.fit(X_trn, y_trn)
            oof_preds[val_idx] = model.predict(X_val)
            test_preds += model.predict(X_test) / skf.n_splits
            # fold_score = roc_auc_score(y_val, oof_preds[val_idx])
            # val_scores.append(fold_score)
            # if verbose:
            #     print(f"Fold {idx}/{n_splits}: {fold_score:.6f}", end=" | ")
                # print(f"Best Alpha: {model.alpha_:.4f}")
            # oof_alphas.append(model.alpha_)
        elif clf == "lasso":
            model = Lasso(
                alpha=alpha_C,
                max_iter=10_000,
                random_state=101,
            )
            model.fit(X_trn, y_trn)
            oof_preds[val_idx] = model.predict(X_val)
            test_preds += model.predict(X_test) / skf.n_splits
            # fold_score = roc_auc_score(y_val, oof_preds[val_idx])
            # val_scores.append(fold_score)
            # if verbose:
            #     print(f"Fold {idx}/{n_splits}: {fold_score:.6f}", end=" | ")
                # print(f"Best Alpha: {model.alpha_:.4f}")
            # oof_alphas.append(model.alpha_)
        elif clf == "logit":
            model = LogisticRegression(
                C=alpha_C,
                max_iter=10_000,
                random_state=101,
                n_jobs=-1,
            )
            model.fit(X_trn, y_trn)
            oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
            test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits
            # fold_score = roc_auc_score(y_val, oof_preds[val_idx])
            # val_scores.append(fold_score)
            # if verbose:
            #     print(f"Fold {idx}/{n_splits}: {fold_score:.6f}", end=" | ")
                # print(f"Best C_: {model.C_[0]:.4f}")
            # oof_alphas.append(model.C_[0])
        elif clf == "ridge":
            model = Ridge(
                alpha=alpha_C,
                random_state=101,
            )
            model.fit(X_trn, y_trn)
            oof_preds[val_idx] = model.predict(X_val)
            test_preds += model.predict(X_test) / skf.n_splits
            # fold_score = roc_auc_score(y_val, oof_preds[val_idx])
            # val_scores.append(fold_score)
            # if verbose:
            #     print(f"Fold {idx}/{n_splits}: {fold_score:.6f}", end=" | ")
                # print(f"Best Alpha: {model.alpha_}")
            # oof_alphas.append(model.alpha_)
        else:
            raise ValueError("Model type not recognized! Please select an appropriate model to train.")

        fold_score = roc_auc_score(y_val, oof_preds[val_idx])
        val_scores.append(fold_score)
        if verbose:
            print(f"Fold {idx}/{n_splits}: {fold_score:.6f}", end=" | ")

    # test_preds /= skf.n_splits

    final_score = np.round(roc_auc_score(y_true, oof_preds), 6)
    avg_score = np.round(np.mean(val_scores), 6)
    std_score = np.round(np.std(val_scores), 6)
    print(f"\n{'='*50}")
    print(f"✓ {clf.title():<7} AUC: {final_score} [{(time()-t0)/60:.2g}mins]")
    print(f"✓ Average AUC: {avg_score} ± {std_score}")
    print("="*50)

    return test_preds, oof_preds, final_score

print("Stacker function ready!")

In [ ]:
oof_df

In [ ]:
pred_df.isna().sum().sum(), oof_df.isna().sum().sum()

In [ ]:
# n_selected = [
#     'realmlp_91924.csv', 'catV1_bayes__919199', 'LGBv1GOSS__918284', 'LGBv2BAG__918642',
#     'XGBv1grad__917515', 'pytabV1_realMLP_918065', 'LGBv8BAG__917359', 'LGBv9BAG__918109',
#     'hill_9175', 'LGBv11BAG__918121', 'kerasV0_91524', 'histV1__loss_917801', 'bartzV1_wbart_916967',
#     'XGBv1loss__917903', 'catV1_loss__91885', 'cat2V1_loss__917702', 'LGBv4BAG__917765',
#     'LGBv7GOSS__917588', 'LGBv3GOSS__917564', 'LGBv3BAG__917314', 'LGBv7BAG__917208', 'LGBv5BAG__91632',
#     'LGBv10GOSS__917361', 'LGBv6GOSS__916954', 'histV1__roc_auc_917785', 'LGBv2GOSS__918589',
#     'catV1_bern__918788', 'LGBv10BAG__916928', 'LGBv12GOSS__917503', 'LGBv5GOSS__916459', 'stack_91751',
#     'LGBv8GOSS__917669', 'h2oV0__916364', 'lgb2v1_bag_127_007_917269', 'lgb2v1_goss_127_005_917231',
#     'h2oV1_20_91791', 'catV2_ord_bayes__919121', 'catV2_loss__918846', 'h2oV2_20_91769', 'catV2_bern__918811',
#     'catV2_bayes__919243', 'lgbv0_bag_256_003_919176', 'lgbvv1_goss_6_006_919182', 'lgbv00_goss_256_003_919215',
#     'lgbvv1_bag_5_01_919045', 'lgbv00_bag_256_003_919127', 'lgbvv1_bag_6_006_919057', 'lgbv0_goss_256_003_919294',
#     'lgbvv1_goss_4_03_918975', 'lgbvv1_bag_5_03_918996', 'lgbvv3_bag_6_007_91906', 'lgbvv2_bag_5_01_919091',
#     'lgbvv2_goss_5_01_919108', 'lgbvv2_goss_6_007_91919', 'lgbvv1_goss_5_01_919099', 'lgbvv1_goss_5_03_919067',
#     'lgbvv1_bag_4_03_91896', 'lgbvv2_bag_6_007_919088', 'lgbvv3_goss_6_007_919236', 'lgbvv4_bag_6_007_919088',
#     'lgbvv4_goss_6_005_919267', 'lgbvv3_goss_5_01_919167', 'lgbvv4_goss_6_007_919247', 'lgbvv4_bag_6_005_919093',
#     'lgbvv3_bag_6_007_918412', 'lgbvv3_goss_6_007_918642', 'lgbvv5_bag_6_007_919078', 'lgbvv6_goss__005_919268',
#     'lgbvv7_bag_005_918934', 'lgbvv8_goss_256_003_919271', 'lgbvv5_goss_6_007_91923', 'lgbvv5_goss_6_005_919259',
#     'lgbvv8_bag_256_003_918963', 'lgbvv6_bag__005_918959', 'lgbvv8_bag_127_005_919019', 'logV5_ohe_cats_first_914921',
#     'lgbvv9_goss_256_005_919311', 'lgbvv8_goss_127_003_919309', 'logV1_ohe_cats_None_916667',
#     'lgbvv9_goss_256_003_919324', 'lgbvv8_goss_127_005_919302', 'lgbvv8_bag_127_003_919048',
#     'pytabV2_cat_Plain_917676', 'pytabV2_tabM_918874', 'logV5_ohe_cats_None_91492', 'pytabV2_cat_Ordered_917682',
#     'pytabV2_mlpPLR_917201', 'pytabV2_xgb_919243', 'logV5_ohe_cats_None_915585', 'pytabV2_realMLP_918141',
#     'pytabV2_lgb_919246', 'tabnetV2__auc_916681', 'pytabV2_resRTDL_917408', 'tabnetV1_auc_916734',
#     'tabnetV0_logloss_912635', 'tabnetV1_logloss_916624', 'tabnetV2__logloss_916788', 'tabnetV1_auc_916792',
#     'tabnetV1_logloss_91688', 'tabnetV3_auc_916912', 'tabnetV3_logloss_916771', 'tabnetV3_logloss_916319',
#     'tabnetV2_auc_915228', 'tabnetV4_logloss_916102', 'tabnetV2_auc_916845', 'tabnetV4_auc_916248',
#     'tabnetV2_logloss_916684', 'tabnetV2_logloss_917032', 'xLearn_v1_ffm_918503', 'xLearn_v1_fm_917331',
#     'tabnetV4_logloss_916769', 'tabnetV5_auc_916916', 'tabnetV4_auc_916518', 'tabnetV6_logloss_916461',
#     'tabnetV5_auc_916478', 'xLearn_v2_ffm_918475', 'xLearn_v1_fm_917328', 'xLearn_v2_linear_91462',
#     'xgb2v2_gbt_005_916962', 'xLearn_v1_linear_915363', 'xLearn_v2_fm_917431', 'xLearn_v1_linear_915355',
#     'xgbV3gbt_01_918995', 'xgbV2grad_4_918157', 'xgb2v2_loss_005_916826', 'xgbV2gbt_4_918308',
#     'xgbvv1gbt_005_918873', 'ydfV0_global_RANDOM_918172', 'xgbV3grad_005_918818', 'xgbvv2grad_005_919039',
#     'xgbV3gbt_005_918973', 'xgbvv2loss_005_919074', 'xgbvv2gbt_005_91914', 'xgbvv1loss_005_91874', 
#     'xgbvv1grad_005_918888', 'ydfV0_local_CART_914746', 'ydfV1_global_CART_917472', 'ydfV0_local_CART_914746_0',
#     'ydfV1_global_RANDOM_91748', 'ydfV1_local_RANDOM_917848', 'ydfV0_global_CART_918141', 'ydfV1_local_CART_917849',
#     'ydfV0_global_CART_918141_0', 'realmlp_9194', 'Xgb_tabM_pseudo_919078', 'Xgb_tabM_pseudo_91882',
#     'Trial_2.0_917778', 'Trial_2.0_917439', 'realmlp_91941',
# ] # 9198919

# print('Old total:', len(n_selected))

# # for c in ['pytabV2_cat_Plain_917676', 'ydfV0_local_CART_914746', 'lgbv0_goss_256_003_919294',
# #  'xgbvv2grad_005_919039', 'catV2_loss__918846']:
# #     if c in n_selected:
# #         print(f"Found {c}. Removed!")
# #         n_selected.remove(c)

# # print('New total:', len(n_selected))

In [ ]:
!rm -r /kaggle/working

In [ ]:
np.linspace(0.01, 0.001, 7)

In [ ]:
%%time

model = "logit"

if model == "logit":
    alpha_Cs = [1.0, 0.1, 0.03] # , 0.005
elif model == "ridge":
    alpha_Cs = [100.0, 1000.0, 10000.0]
elif model == "lasso":
    alpha_Cs = [0.01, 0.001, 0.0001]
elif model == "elastic":
    alpha_Cs = [0.01, 0.001, 0.0001]
elif model == "mlp":
    alpha_Cs = [0.1, 0.01, 0.001]
else:
    alpha_Cs = ["_"]

to_logit = True
scaling  = True
ranking  = False

# - Store train results - 
_scores    = []
oof_preds  = []
test_preds = []
cv_results = {}

# if len(alpha_Cs) > 4:
#     print(f"C values more than 4, selecting last 4 values for training")
#     alpha_Cs = alpha_Cs[-4:]

for i, c in enumerate(alpha_Cs):
    _test, _oof, _score = train_meta(
        clf=model,
        oof_df=oof_df,
        pred_df=pred_df,
        y_true=y_true,
        alpha_C=c,
        to_logits=to_logit,
        apply_scaler=scaling,
        rank_first=ranking,
        # verbose=False,
    )
    _scores.append(_score)
    oof_preds.append(_oof)
    test_preds.append(_test)
    print()

plt.figure(figsize=(12.5, 4))
sns.lineplot(_scores, marker="o", linestyle="-.")

for i, s in enumerate(_scores):
    plt.text(float(i), s+1e-7, s, ha="center", va="baseline")

plt.title(f"{model.title()} alpha(C) scores -> (5-fold each)", weight="semibold")
plt.xticks(range(len(alpha_Cs)), alpha_Cs)
plt.xlabel("alpha(C)")
plt.tight_layout()
plt.show()

print()
## -----------------------------------------------------------------------------
plt.figure(figsize=(12.5, 4))
for i, (c, s, o, p) in enumerate(zip(alpha_Cs, _scores, oof_preds, test_preds)):
    plt.subplot(1, len(alpha_Cs), i+1)
    sns.histplot(p, label="test_predictions", bins=50)
    plt.title(f"{model.title()} alpha_C({c})={s}", weight="semibold") 
    plt.legend() 
    plt.tight_layout()

    n = f"submit_{model}({len(MODEL_COLS)})_{FOLDS}cvStack_C{c}_{str(s).split('.')[1]}.csv"
    submit[TARGET] = p
    submit.to_csv(n, index=False)

    # np.save(f"oof_{model}_cvStack_C{c}_{str(s).split('.')[1]}.npy", o)
    # np.save(f"test_{model}_cvStack_C{c}_{str(s).split('.')[1]}.npy", p)

    cv_results[f"{c}__{s}"] = p

plt.show()
print()
## -----------------------------------------------------------------------------
cv_results = pd.DataFrame(cv_results)
cv_results

In [ ]:
cv_results.corr()

In [ ]:
# (
#     cv_results["0.01__0.969352"] * 0.8
#     + cv_results["0.001__0.969388"] * 0.15
#     + cv_results["0.0001__0.969379"] * 0.05
# )

In [ ]:
# n = f"submit_mlp_weighted_3_models.csv"

# submit[TARGET] = (
#     cv_results["0.01__0.969352"] * 0.7
#     + cv_results["0.001__0.969388"] * 0.25
#     + cv_results["0.0001__0.969379"] * 0.05
# )
# submit.to_csv(n, index=False)

# submit

In [ ]:
# !rm -r /kaggle/working 

In [ ]:
full_model_preds = {}
full_models = []

full_model_preds, full_models 

In [ ]:
## -- Train full data --
log_clf = LogisticRegression(
    C=0.03,
    max_iter=10_000,
    random_state=101,
    n_jobs=-1,
)

mlp_clf = MLPClassifier(
    alpha=0.01, # default=0.0001
    max_iter=50,
    # tol=0.0001,
    hidden_layer_sizes=(oof_df.shape[1],), # default=(100,)
    random_state=101,
    # batch_size=128, # default='auto'
    activation='relu', # {'identity', 'logistic', 'tanh', 'relu'}, default='relu'
    solver='adam', # {'lbfgs', 'sgd', 'adam'}, default='adam'
    learning_rate='constant',
    learning_rate_init=0.001, # default=0.001
    power_t=0.5,
    shuffle=True,
    verbose=False,
    warm_start=False,
    momentum=0.9,
    nesterovs_momentum=True,
    # early_stopping=True, # default=False
    validation_fraction=0.1,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-08,
    n_iter_no_change=10,
    max_fun=15000,
)

full_clfs = [
    ("logit", log_clf),
    # ("mlp", mlp_clf),
]

for i, (name, model) in enumerate(full_clfs):
    t0=time()
    print(f"\n+++++ {name} fit on full data training +++++")
    
    print("  ++ Probs to logits...")
    oofs  = prob_to_logit(oof_df.to_numpy())
    preds = prob_to_logit(pred_df.to_numpy())
    
    # print("  ++ Scaling features... ")
    # scaler = StandardScaler()
    # oofs   = scaler.fit_transform(oofs, y_true)
    # preds  = scaler.transform(preds)

    print(f"oof={oofs.shape}, test={preds.shape}")

    model.fit(oofs, y_true)
    full_oof_preds = model.predict_proba(oofs)[:, 1]
    full_score = np.round(roc_auc_score(y_true, full_oof_preds), 6)
    print(f"{name} AUC = {full_score}")

    ## -- Save test --
    full_preds = model.predict_proba(preds)[:, 1]

    full_model_preds[f"{name}_{full_score}"] = full_preds
    # full_models.append(model)

    print(f" ⧗ {name} full train complete - 💾 test predictions saved!")

    n = f"submit_{name}({len(MODEL_COLS)})_FULL_{str(full_score).split('.')[1]}.csv"
    submit[TARGET] = full_preds
    submit.to_csv(n, index=False)

    print(f"{name} full submission sxported -> ", n)
    dura = (time()-t0) / 60
    print(f"{name} duration: {dura:.2f} mins")

    if 'logit' in name:
        ## -- Plot coefs --
        full_coefs = dict(zip(oof_df.columns, model.coef_[0]))
        full_table = pd.Series(full_coefs, name='score').abs().sort_values(ascending=False)
        full_cols  = full_table[full_table.values > 0].index.tolist()
        
        plt.figure(figsize=(12.5, max(6, len(full_table) * 0.25)))
        sns.barplot(full_table, orient="h", palette='Blues_r')
        plt.title(f"{name.upper()} coefs | Top cols (abs) = {oofs.shape[1]}", size=14, weight='bold')

        plt.savefig(f"{name.upper()}_full_coefs")
        plt.tight_layout()
        plt.show()
    else:
        ## -- Plot coefs --
        full_coefs = dict(zip(oof_df.columns, model.coefs_[0][0]))
        full_table = pd.Series(full_coefs, name='score').abs().sort_values(ascending=False)
        full_cols  = full_table[full_table.values > 0].index.tolist()
        
        plt.figure(figsize=(12.5, max(6, len(full_table) * 0.25)))
        sns.barplot(full_table, orient="h", palette='Blues_r')
        plt.title(f"{name.upper()} coefs | Top cols (abs) = {oofs.shape[1]}", size=14, weight='bold')

        plt.savefig(f"{name.upper()}_full_coefs")
        plt.tight_layout()
        plt.show()

full_preds_df = pd.DataFrame(full_model_preds)
full_preds_df # 0.946357

In [ ]:
full_preds_df["ranked_avg"] = full_preds_df.rank(method="average", pct=True).mean(axis=1)

n = f"submit_ranked_avg_FULL_mlp_logit.csv"
submit[TARGET] = full_preds_df["ranked_avg"]
submit.to_csv(n, index=False)

print(f"💾 Saved -> {n}")

full_preds_df

In [ ]:
# blend_output = (0.6 * full_preds_df["mlp_0.969481"]) + (0.4 * full_preds_df["logit_0.969421"])

# n = f"submit_blend_FULL_mlp_logit.csv"
# submit[TARGET] = blend_output
# submit.to_csv(n, index=False)

# submit

In [ ]:
# !rm -r /kaggle/working 

In [ ]:
oof_df

In [ ]:
pred_df

In [ ]:
## -- Forward Model Selection --
ROUNDS       = 3
FINAL_SCORES = {}
FINAL_MODELS = {}
search_times = []

rng = np.random.default_rng()

for i in range(ROUNDS):
    t0 = time()
    best_score  = 0
    best_models = []

    all_models = MODEL_COLS.copy()
    models_id  = [_ for _ in range(len(all_models))]
    ## -----------------------------------------------
    shuffle_id = rng.permutation(models_id)
    starter_id = [int(a) for a in shuffle_id][0]
    ## -----------------------------------------------
    init_model = all_models[starter_id]

    _id = models_id.pop(starter_id)
    other_models = [all_models[_] for _ in models_id]
    other_models = rng.permutation(other_models).tolist()

    print(f"\n+++++ Round {i+1}/{ROUNDS} • Starting with {init_model}")
    
    pbar = tqdm(enumerate(other_models), total=len(other_models))

    for j, m in pbar:
        total_models = best_models + [init_model, m]

        print(f"\t{j+1}/{len(all_models)} ++ Added {m} *****")
        print('-'*70)
        _, __, _score = train_meta(
            clf="logit",
            features=total_models,
            oof_df=oof_df,
            pred_df=pred_df,
            y_true=y_true,
            alpha_C=0.03,
            to_logits=True,
            apply_scaler=True,
            # rank_first=True,
            verbose=False,
        )

        if _score > best_score:
            print(f"{CFG['GREEN']} • {m}: ⤴ {_score}{CFG['RESET']}")
            best_score = _score
            best_models.append(m)
        else:
            print(f"{CFG['RED']} • {m}: ⤵ {_score}{CFG['RESET']}")

        # print()
        pbar.set_description(f"Current score: {best_score:.6f}")

    best_models.append(init_model)
    print(f"\nBest score: {best_score:.6f}")
    print(f"Best models: {len(best_models)}\n{best_models}\n")

    FINAL_SCORES[f"round_{i}"] = best_score
    FINAL_MODELS[f"round_{i}"] = best_models

    duration = np.round((time()-t0)/60, 2)
    search_times.append(duration)
    print(f"{duration:.2f} mins\n")

In [ ]:
for d in search_times:
    print(f"{d:.4f} mins")

In [ ]:
top_name, top_score = max(FINAL_SCORES.items(), key=lambda item: item[1])

plt.figure(figsize=(12.5, 4)) 
sns.lineplot(FINAL_SCORES, marker="o", linestyle="--")

for i, s in enumerate(FINAL_SCORES.values()):
    plt.text(float(i), s+1e-6, s, ha="center", va="baseline")

plt.title(f"Best {top_name} | Score = {top_score}")
plt.ylabel("score")
plt.xlabel("Rounds")
plt.show()

In [ ]:
print(FINAL_SCORES)

best_score = 0
best_id = 0
for k, v in FINAL_SCORES.items():
    if v > best_score:
        best_score = v
        best_id = k
    else:
        continue
        
top_models = FINAL_MODELS[best_id]

print()
print(f"🔥 Best round: {best_id} | Score: {best_score} | Features: {len(top_models)}/{len(MODEL_COLS)}")

In [ ]:
print()
print(f"🔥 Best round: {best_id} | Score: {best_score} | Features: {len(top_models)}/{len(MODEL_COLS)}")

top_models

In [ ]:
# !rm -r /kaggle/working

In [ ]:
%%time

model = "logit"

if model == "logit":
    alpha_Cs = [1.0, 0.1, 0.03] # , 0.005
elif model == "ridge":
    alpha_Cs = [100.0, 1000.0, 10000.0]
elif model == "lasso":
    alpha_Cs = [0.01, 0.001, 0.0001]
elif model == "elastic":
    alpha_Cs = [0.01, 0.001, 0.0001]
elif model == "mlp":
    alpha_Cs = [0.1, 0.01, 0.001]
else:
    alpha_Cs = ["_"]

to_logit = True
scaling  = True
ranking  = False

# - Store train results - 
_scores    = []
oof_preds  = []
test_preds = []
cv_results = {}

# if len(alpha_Cs) > 4:
#     print(f"C values more than 4, selecting last 4 values for training")
#     alpha_Cs = alpha_Cs[-4:]

for i, c in enumerate(alpha_Cs):
    _test, _oof, _score = train_meta(
        clf=model,
        features=top_models,
        oof_df=oof_df,
        pred_df=pred_df,
        y_true=y_true,
        alpha_C=c,
        to_logits=to_logit,
        apply_scaler=scaling,
        rank_first=ranking,
        # verbose=False,
    )
    _scores.append(_score)
    oof_preds.append(_oof)
    test_preds.append(_test)
    print()

plt.figure(figsize=(12.5, 4))
sns.lineplot(_scores, marker="o", linestyle="-.")

for i, s in enumerate(_scores):
    plt.text(float(i), s+1e-7, s, ha="center", va="baseline")

plt.title(f"{model.title()} alpha(C) scores -> (5-fold each)", weight="semibold")
plt.xticks(range(len(alpha_Cs)), alpha_Cs)
plt.xlabel("alpha(C)")
plt.tight_layout()
plt.show()

print()
## -----------------------------------------------------------------------------
plt.figure(figsize=(12.5, 4))
for i, (c, s, o, p) in enumerate(zip(alpha_Cs, _scores, oof_preds, test_preds)):
    plt.subplot(1, 1*len(alpha_Cs), i + 1)
    sns.histplot(p, label="test_predictions", bins=50)
    plt.title(f"{model.title()} alpha_C({c}) = {s}", weight="semibold") 
    plt.legend() 
    plt.tight_layout()

    n = f"submit_{model}({len(top_models)})_{FOLDS}cvStack_C{c}_{str(s).split('.')[1]}.csv"
    submit[TARGET] = p
    submit.to_csv(n, index=False)

    # np.save(f"oof_{model}_cvStack_C{c}_{str(s).split('.')[1]}.npy", o)
    # np.save(f"test_{model}_cvStack_C{c}_{str(s).split('.')[1]}.npy", p)
    cv_results[f"{c}__{s}"] = p
    
plt.show()

print()
## -----------------------------------------------------------------------------
cv_results = pd.DataFrame(cv_results)
cv_results

In [ ]:
## -- Train full data --
log_clf = LogisticRegression(
    C=0.03,
    max_iter=10_000,
    random_state=101,
    n_jobs=-1,
)

mlp_clf = MLPClassifier(
    alpha=0.01, # default=0.0001
    max_iter=50,
    # tol=0.0001,
    hidden_layer_sizes=(oof_df.shape[1],), # default=(100,)
    random_state=101,
    # batch_size=128, # default='auto'
    activation='relu', # {'identity', 'logistic', 'tanh', 'relu'}, default='relu'
    solver='adam', # {'lbfgs', 'sgd', 'adam'}, default='adam'
    learning_rate='constant',
    learning_rate_init=0.001, # default=0.001
    power_t=0.5,
    shuffle=True,
    verbose=False,
    warm_start=False,
    momentum=0.9,
    nesterovs_momentum=True,
    early_stopping=True, # default=False
    validation_fraction=0.1,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-08,
    n_iter_no_change=10,
    max_fun=15000,
)

full_clfs = [
    ("logit", log_clf),
    # ("mlp", mlp_clf),
]

for i, (name, model) in enumerate(full_clfs):
    t0=time()
    print(f"\n+++++ {name} fit on full data: SELECTED FEATURES {len(top_models)} +++++")
    
    print("  ++ Probs to logits...")
    oofs  = prob_to_logit(oof_df[top_models].to_numpy())
    preds = prob_to_logit(pred_df[top_models].to_numpy())
    
    print("  ++ Scaling features... ")
    scaler = StandardScaler()
    oofs   = scaler.fit_transform(oofs, y_true)
    preds  = scaler.transform(preds)

    print(f"oof= {oofs.shape}, test= {preds.shape}")

    model.fit(oofs, y_true)
    full_oof_preds = model.predict_proba(oofs)[:, 1]
    full_score = np.round(roc_auc_score(y_true, full_oof_preds), 6)
    print(f"{name} AUC = {full_score}")

    ## -- Save test --
    full_preds = model.predict_proba(preds)[:, 1]

    full_model_preds[f"{name}_{full_score}"] = full_preds
    # full_models.append(model)

    print(f"⧗ {name} full train complete - 💾 test predictions saved!")

    n = f"submit_{name}_FULL({len(top_models)})_{str(full_score).split('.')[1]}.csv"
    submit[TARGET] = full_preds
    submit.to_csv(n, index=False)

    print(f"💾 {name} full submission -> ", n)
    dura = (time()-t0) / 60
    print(f"{name} duration: {dura:.2f} mins")

    if 'logit' in name:
        ## -- Plot coefs --
        full_coefs = dict(zip(oof_df.columns, model.coef_[0]))
        full_table = pd.Series(full_coefs, name='score').abs().sort_values(ascending=False)
        full_cols  = full_table[full_table.values > 0].index.tolist()
        
        plt.figure(figsize=(12.5, max(5, len(full_table) * 0.25)))
        sns.barplot(full_table, orient="h", palette='Blues_r')
        plt.title(f"Logistic coefs | Top cols (abs) = {oofs.shape[1]}")

        plt.savefig(f"{name}_full_coefs")
        plt.tight_layout() 
        plt.show()

full_preds_df = pd.DataFrame(full_model_preds)
full_preds_df

## ENSEMBLE 2: HILL CLIMBING

In [ ]:
# # start = 180
# # up_to = -1

# load_hilclimb = []
# extra_feat = []
# for c in oof_df.columns:
#     if 'hillclimb' in c:
#         load_hilclimb.append(c)
#     elif 'realmlp_91941' in c:
#         extra_feat.append(c)
#     # elif 'realmlp_9194' in c:
#     #     extra_feat.append(c)

# # n_features = [col for col in oof_df.columns[start:]]
# # # n_features += n_selected

# # print(f"Selected Total: {len(n_selected)}")
# # print(f"Added 10 Total: {len(n_features)}")

# extra_feat, load_hilclimb

In [ ]:
# ## -- Load saved hillclimb oof/test
# hc_oof  = pd.Series(np.load('/kaggle/working/oof_HC_cvStack_148_919892.npy'), name='hillclimb')
# hc_test = pd.Series(np.load('/kaggle/working/test_HC_cvStack_148_919892.npy'), name='hillclimb')

# hc_oof.head(), hc_test.head()

In [ ]:
%%time
# pd.concat([oof_df[extra_feat], oof_df[load_hilclimb]], axis=1),
# pd.concat([pred_df[extra_feat], pred_df[load_hilclimb]], axis=1),

test_hillclimb, oof_hillclimb= climb_hill(
                               train = train,
                              target = TARGET,
                         oof_pred_df = oof_df,
                        test_pred_df = pred_df,
                           objective = "maximize",
                         eval_metric = partial(roc_auc_score),
                    # negative_weights = True,
                    return_oof_preds = True,
                           precision = 0.01,
                           plot_hill = True,
                           # plot_hist = True,
    )

In [ ]:
# n_selected = [
#     'realmlp_91924.csv', 'catV1_bayes__919199', 'LGBv1GOSS__918284', 'LGBv2BAG__918642',
#     'XGBv1grad__917515', 'pytabV1_realMLP_918065', 'LGBv8BAG__917359', 'LGBv9BAG__918109',
#     'hill_9175', 'LGBv11BAG__918121', 'kerasV0_91524', 'histV1__loss_917801', 'bartzV1_wbart_916967',
#     'XGBv1loss__917903', 'catV1_loss__91885', 'cat2V1_loss__917702', 'LGBv4BAG__917765',
#     'LGBv7GOSS__917588', 'LGBv3GOSS__917564', 'LGBv3BAG__917314', 'LGBv7BAG__917208', 'LGBv5BAG__91632',
#     'LGBv10GOSS__917361', 'LGBv6GOSS__916954', 'histV1__roc_auc_917785', 'LGBv2GOSS__918589',
#     'stack_91751', 'catV1_bern__918788', 'LGBv10BAG__916928', 'LGBv12GOSS__917503', 'LGBv5GOSS__916459',
#     'LGBv8GOSS__917669', 'h2oV0__916364', 'lgb2v1_bag_127_007_917269', 'lgb2v1_goss_127_005_917231',
#     'h2oV1_20_91791', 'catV2_ord_bayes__919121', 'catV2_loss__918846', 'h2oV2_20_91769', 'catV2_bern__918811',
#     'catV2_bayes__919243', 'lgbv0_bag_256_003_919176', 'lgbvv1_goss_6_006_919182', 'lgbv00_goss_256_003_919215',
#     'lgbvv1_bag_5_01_919045', 'lgbv00_bag_256_003_919127', 'lgbvv1_bag_6_006_919057', 'lgbv0_goss_256_003_919294',
#     'lgbvv1_goss_4_03_918975', 'lgbvv1_bag_5_03_918996', 'lgbvv3_bag_6_007_91906', 'lgbvv2_bag_5_01_919091',
#     'lgbvv2_goss_5_01_919108', 'lgbvv2_goss_6_007_91919', 'lgbvv1_goss_5_01_919099', 'lgbvv1_goss_5_03_919067',
#     'lgbvv1_bag_4_03_91896', 'lgbvv2_bag_6_007_919088', 'lgbvv3_goss_6_007_919236', 'lgbvv4_bag_6_007_919088',
#     'lgbvv4_goss_6_005_919267', 'lgbvv3_goss_5_01_919167', 'lgbvv4_goss_6_007_919247', 'lgbvv4_bag_6_005_919093',
#     'lgbvv3_bag_6_007_918412', 'lgbvv3_goss_6_007_918642', 'lgbvv5_bag_6_007_919078', 'lgbvv6_goss__005_919268',
#     'lgbvv7_bag_005_918934', 'lgbvv8_goss_256_003_919271', 'lgbvv5_goss_6_007_91923', 'lgbvv5_goss_6_005_919259',
#     'lgbvv8_bag_256_003_918963', 'lgbvv6_bag__005_918959', 'lgbvv8_bag_127_005_919019', 'logV5_ohe_cats_first_914921',
#     'lgbvv9_goss_256_005_919311', 'lgbvv8_goss_127_003_919309', 'logV1_ohe_cats_None_916667',
#     'lgbvv9_goss_256_003_919324', 'lgbvv8_goss_127_005_919302', 'lgbvv8_bag_127_003_919048',
#     'pytabV2_cat_Plain_917676', 'pytabV2_tabM_918874', 'logV5_ohe_cats_None_91492', 'pytabV2_cat_Ordered_917682',
#     'pytabV2_mlpPLR_917201', 'pytabV2_xgb_919243', 'logV5_ohe_cats_None_915585', 'pytabV2_realMLP_918141',
#     'pytabV2_lgb_919246', 'tabnetV2__auc_916681', 'pytabV2_resRTDL_917408', 'tabnetV1_auc_916734',
#     'tabnetV0_logloss_912635', 'tabnetV1_logloss_916624', 'tabnetV2__logloss_916788', 'tabnetV1_auc_916792',
#     'tabnetV1_logloss_91688', 'tabnetV3_auc_916912', 'tabnetV3_logloss_916771', 'tabnetV3_logloss_916319',
#     'tabnetV2_auc_915228', 'tabnetV4_logloss_916102', 'tabnetV2_auc_916845', 'tabnetV4_auc_916248',
#     'tabnetV2_logloss_916684', 'tabnetV2_logloss_917032', 'xLearn_v1_ffm_918503', 'xLearn_v1_fm_917331',
#     'tabnetV4_logloss_916769', 'tabnetV5_auc_916916', 'tabnetV4_auc_916518', 'tabnetV6_logloss_916461',
#     'tabnetV5_auc_916478', 'xLearn_v2_ffm_918475', 'xLearn_v1_fm_917328', 'xLearn_v2_linear_91462',
#     'xgb2v2_gbt_005_916962', 'xLearn_v1_linear_915363', 'xLearn_v2_fm_917431', 'xLearn_v1_linear_915355',
#     'xgbV3gbt_01_918995', 'xgbV2grad_4_918157', 'xgb2v2_loss_005_916826', 'xgbV2gbt_4_918308',
#     'xgbvv1gbt_005_918873', 'ydfV0_global_RANDOM_918172', 'xgbV3grad_005_918818', 'xgbvv2grad_005_919039',
#     'xgbV3gbt_005_918973', 'xgbvv2loss_005_919074', 'xgbvv2gbt_005_91914', 'xgbvv1loss_005_91874', 
#     'xgbvv1grad_005_918888', 'ydfV0_local_CART_914746', 'ydfV1_global_CART_917472', 'ydfV0_local_CART_914746_0',
#     'ydfV1_global_RANDOM_91748', 'ydfV1_local_RANDOM_917848', 'ydfV0_global_CART_918141', 'ydfV1_local_CART_917849',
#     'ydfV0_global_CART_918141_0', 'realmlp_9194', 'Xgb_tabM_pseudo_919078', 'Xgb_tabM_pseudo_91882',
#     'Trial_2.0_917778', 'Trial_2.0_917439', 
# ] # 9198916

In [ ]:
hc_score = np.round(roc_auc_score(y_true, oof_hillclimb), 6)

print(f"-> Hillclimb score: {hc_score}")
print(f"""\n 
-> TRUE Avg.: {np.mean(y_true):.6f}
-> OOFS Avg.: {np.mean(oof_hillclimb):.6f} 
-> TEST Avg.: {np.mean(test_hillclimb):.6f}
    """)

# np.save(f"oof_hillclimb_cvStack_{str(hc_score).split('.')[1]}.npy", oof_hillclimb)
# np.save(f"test_hillclimb_cvStack_{str(hc_score).split('.')[1]}.npy", test_hillclimb)

## -- SUBMISSION FILE --
n = f"submit_hillclimb({len(MODEL_COLS)})_{str(hc_score).split('.')[1]}.csv"
submit[TARGET] = test_hillclimb
submit.to_csv(n, index=False)

print(f"Hillclimb submission file exported -> {n}\n")
submit

## ENSEMBLE 3: OPTIMIZED BLENDING WITH SCIPY

In [ ]:
from scipy.optimize import minimize
np.random.seed(101)

In [ ]:
columns = [
    'ridge_'+str(ridge_score).split('.')[1],
    'linreg_'+str(linreg_score).split('.')[1],
    'log_'+str(logit_score).split('.')[1],
    # 'lasso_'+str(lasso_score).split('.')[1],
    # 'elastic_'+str(elastic_score).split('.')[1],
    'bayes_'+str(bayes_score).split('.')[1],
    # 'mlp_'+str(mlp_score).split('.')[1],
    'hill_'+str(hc_score).split('.')[1],
    # 'hill_'+str(0.919892).split('.')[1],
]
oof_predictions = [
    oof_ridge,
    oof_linreg,
    oof_logit,
    # oof_lasso,
    # oof_elastic,
    oof_bayes,
    # oof_mlp,
    oof_df['hillclimb_cvStack_149_919892'].to_numpy(),
    # oof_hillclimb,
]
test_predictions = [
    test_ridge,
    test_linreg,
    test_logit,
    # test_lasso,
    # test_elastic,
    test_bayes,
    # test_mlp,
    pred_df['hillclimb_cvStack_149_919892'].to_numpy(),
    # test_hillclimb,
]

train_final = pd.DataFrame(dict(zip(columns, oof_predictions)))
test_final  = pd.DataFrame(dict(zip(columns, test_predictions)))

train_final.shape, test_final.shape

In [ ]:
display(train_final.head())
print() 
display(test_final .head())

In [ ]:
blend_train = []
blend_test = []

for col in train_final:
    blend_train.append(train_final[col])
blend_train = np.array(blend_train)

for col in test_final:
    blend_test.append(test_final[col])
blend_test = np.array(blend_test)

a = 0.12345
print("{0:.2f}".format(a))

In [ ]:
## -- BLOCK 2 - AUC-based objective and optimization
def roc_min_func(weights):
    final_prediction = np.tensordot(weights, blend_train, axes=(0, 0))
    return roc_auc_score(y_true, final_prediction)
    
print('\n Finding Blending Weights (AUC objective) *****\n')
res_list = []
weights_list = []

n_models = blend_train.shape[0]

for k in range(20):  # -> Optimization rounds
    starting_values = np.random.uniform(size=n_models)
    bounds = [(0, 1)] * n_models

    res = minimize(
        roc_min_func,
        starting_values,
        method='L-BFGS-B',
        bounds=bounds,
        options={'disp': False, 'maxiter': 10000},
    )

    res_list.append(res['fun'])
    weights_list.append(res['x'])

    res_fun = format(res['fun'], '.6f')
    items = [format(item, '.6f') for item in res['x']]

    print(
        '{iter}\tScore: {score}\tWeights: {weights}'.format(
            iter=(k + 1),
            score=res_fun,
            weights=' | '.join([str(item) for item in items]),
        )
    )

bestAUC = np.max(res_list)  # AUC: higher is better
bestWght = weights_list[int(np.argmax(res_list))]
weights = bestWght

blend_score = round(bestAUC, 6)

print(f"Best score: {CFG['GREEN']}{blend_score}")

In [ ]:
print(f"Best score: {CFG['GREEN']}{blend_score}")

In [ ]:
# BLOCK 3 - reporting and final model
print(f'Ensemble AUC: {blend_score}')
# print('\n Best Weights: {weights}'.format(weights=bestWght))

train_prices = np.zeros(len(blend_train[0]))
test_prices  = np.zeros(len(blend_test[0]))

print()
print(' ----------------')
print(' Weights  | Model')
print(' ----------------')

for k in range(len(blend_test)):
    print(' %.6f * model-%d' % (weights[k], (k + 1)))
    test_prices += blend_test[k] * weights[k]

for k in range(len(blend_train)):
    train_prices += blend_train[k] * weights[k]

# Optional: normalize weights to sum to 1 (same as before)
bestWght /= np.sum(bestWght)
print()

# BLOCK 4 - optional visualization (unchanged logic)
optimized_oof = np.tensordot(bestWght, train_final, axes=((0), (1)))  # shape (n_samples,)
sns.histplot(optimized_oof, bins=50)
plt.title(f"Optimized Predictions (AUC): {blend_score}")
plt.legend()
plt.show()

In [ ]:
## -- Apply weights to test --
optimized_test = test_final.to_numpy().dot(weights)
submit[TARGET] = optimized_test
submit.to_csv(f"submit_optimized_blend_{str(blend_score).split('.')[1]}.csv", index=False)
submit

In [ ]:
# !rm -r /kaggle/working